In [1]:
import sys

import bibcat

print(sys.executable)
print(bibcat.__file__)


import bibcat.llm.cm as m

m.__file__

/Users/jyoon/micromamba/envs/bibcat/bin/python
/Users/jyoon/GitHub/bibcat/bibcat/__init__.py


'/Users/jyoon/GitHub/bibcat/bibcat/llm/metrics.py'

In [2]:
# %pip uninstall -y bibcat
# %pip uninstall -y bibcat   # run twice to remove duplicate installs if present
# %pip install -e /Users/jyoon/GitHub/bibcat


Found existing installation: bibcat 0.2.4.dev4+g4d0f06342.d20251009
Uninstalling bibcat-0.2.4.dev4+g4d0f06342.d20251009:
  Successfully uninstalled bibcat-0.2.4.dev4+g4d0f06342.d20251009
Note: you may need to restart the kernel to use updated packages.
Found existing installation: bibcat 0.2.6.dev2+g195a2c182
Uninstalling bibcat-0.2.6.dev2+g195a2c182:
  Successfully uninstalled bibcat-0.2.6.dev2+g195a2c182
Note: you may need to restart the kernel to use updated packages.
Obtaining file:///Users/jyoon/GitHub/bibcat
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for bibcat (pyproject.toml) ... done
  Created wheel for bibcat: filename=bibcat-0.2.6.dev14+gc891792c1-0.editable-py3-none-any.whl size=10593 sha256=593a40b5aa9502637a7dae153589dbff36bb9d6ed54a035f2318b79e718a4791
  Stored in directory: /private/var/f

In [3]:
import bibcat.llm.cm as m

print(m.__file__)
# from bibcat.llm.cm import build_samples_for_run

print("ok")

/Users/jyoon/GitHub/bibcat/bibcat/llm/metrics.py
ok


In [ ]:
from pprint import pprint

from bibcat.llm.cm import (
    build_cm_mission_samples_from_run_verdicts,
    compute_confusion,
    compute_metrics,
    extract_samples_and_summary,
)
from bibcat.llm.evaluation_base import build_run_paper_verdicts, build_source_lookup


def sample_key(sample):
    return sample.bibcode, sample.mission


# comparing single run from eval_data vs a run from llm_runs_data for the same missions and run_index
def compare_single_vs_run(eval_data, llm_runs_data, missions, run_index=0, limit=20):
    single_samples, single_summary = extract_samples_and_summary(eval_data, missions)

    source_lookup = build_source_lookup()
    run_verdicts = build_run_paper_verdicts(
        llm_runs_data=llm_runs_data,
        bibcodes=list(eval_data.keys()),
        run_index=run_index,
        source_lookup=source_lookup,
    )
    run_samples = build_cm_mission_samples_from_run_verdicts(
        run_verdicts=run_verdicts,
        missions=missions,
    )

    single_by_key = {sample_key(s): s for s in single_samples}
    run_by_key = {sample_key(s): s for s in run_samples}

    all_keys = sorted(set(single_by_key) | set(run_by_key))

    diffs = []

    for key in all_keys:
        s1 = single_by_key.get(key)
        s2 = run_by_key.get(key)

        if s1 is None or s2 is None:
            diffs.append(
                {
                    "bibcode": key[0],
                    "mission": key[1],
                    "issue": "missing sample in one path",
                    "single_exists": s1 is not None,
                    "run_exists": s2 is not None,
                }
            )
            continue

        if (
            s1.human_raw != s2.human_raw
            or s1.llm_raw != s2.llm_raw
            or s1.human_label != s2.human_label
            or s1.llm_label != s2.llm_label
        ):
            diffs.append(
                {
                    "bibcode": key[0],
                    "mission": key[1],
                    "single": {
                        "human_raw": s1.human_raw,
                        "llm_raw": s1.llm_raw,
                        "human_label": s1.human_label,
                        "llm_label": s1.llm_label,
                    },
                    "run": {
                        "human_raw": s2.human_raw,
                        "llm_raw": s2.llm_raw,
                        "human_label": s2.human_label,
                        "llm_label": s2.llm_label,
                    },
                }
            )

    single_metrics = compute_metrics(compute_confusion(single_samples))
    run_metrics = compute_metrics(compute_confusion(run_samples))

    print("\n=== Summary ===")
    print(f"run_index: {run_index}")
    print(f"single samples: {len(single_samples)}")
    print(f"run samples:    {len(run_samples)}")
    print(f"diff count:     {len(diffs)}")

    print("\n=== Single metrics ===")
    pprint(single_metrics)

    print("\n=== Run metrics ===")
    pprint(run_metrics)

    print(f"\n=== First {limit} diffs ===")
    pprint(diffs[:limit])

    return {
        "single_summary": single_summary,
        "single_metrics": single_metrics,
        "run_metrics": run_metrics,
        "diffs": diffs,
    }

In [ ]:
# Example usage:

import json

eval_file = "/Users/jyoon/asb/bibliography_automation/bibcat_output/output/llms/openai_gpt-4.1-mini/flagship_GS_prompt6_ads-text_n10/flagship_GS_prompt6_ads-text_summary_output_n10_05-07-2026_t0.5.json"
with open(eval_file) as f:
    eval_data = json.load(f)

llm_runs_file = "/Users/jyoon/asb/bibliography_automation/bibcat_output/output/llms/openai_gpt-4.1-mini/flagship_GS_prompt6_ads-text_n10/flagship_GS_prompt6_ads-text_llm_output_n10.json"
with open(llm_runs_file) as f:
    llm_runs_data = json.load(f)

# missions = ["TESS", "HST", "GALEX", "PANSTARRS"]
missions = ["HST"]

result = compare_single_vs_run(
    eval_data=eval_data,
    llm_runs_data=llm_runs_data,
    missions=missions,
    # run_index=0,
)

2026-05-06 19:11:53,079 - bibcat.data.build_dataset - INFO - Loading source dataset: /Users/jyoon/Documents/asb/bibliography_automation/bibcat_datasets//combined_dataset_2025_07_08.json


2026-05-06 19:12:00,470 - bibcat.llm.evaluate - WARNING - No mission output found for 2022JApA...43...86S
2026-05-06 19:12:01,880 - bibcat.llm.evaluate - WARNING - Error processing paper paragraphs: Err: Unrecognized ambig. phrase:
Hubble Diagram scatter
Taken from this text snippet:
Using 56 SNe Ia with NIR data near peak brightness, where the luminosity dispersion is minimal, we found a 35 per cent reduction in Hubble Diagram scatter (i.e. more precise distances) when using SNe Ia as NIR standard candles, relative to conventional optical-only fits to the same SNe..
2026-05-06 19:12:02,522 - bibcat.llm.evaluate - WARNING - Error processing paper paragraphs: Err: Unrecognized ambig. phrase:
other Hubble + Webb surveys
Taken from this text snippet:
This survey has a much shallower depth, but aims to detect ‘an order of magnitude more early Universe galaxies than all other Hubble + Webb surveys combined’ (Kartaltepe et al 2021), due to its very large survey area..
2026-05-06 19:12:03,104


=== Summary ===
run_index: 0
single samples: 120
run samples:    120
diff count:     20

=== Single metrics ===
{'accuracy': 0.8083333333333333,
 'f1': 0.7228915662650603,
 'fn': 17,
 'fnr': 0.3617021276595745,
 'fp': 6,
 'fpr': 0.0821917808219178,
 'precision': 0.8333333333333334,
 'recall': 0.6382978723404256,
 'tn': 67,
 'tnr': 0.9178082191780822,
 'tp': 30,
 'tpr': 0.6382978723404256}

=== Run metrics ===
{'accuracy': 0.8,
 'f1': 0.7142857142857143,
 'fn': 19,
 'fnr': 0.3877551020408163,
 'fp': 5,
 'fpr': 0.07042253521126761,
 'precision': 0.8571428571428571,
 'recall': 0.6122448979591837,
 'tn': 66,
 'tnr': 0.9295774647887324,
 'tp': 30,
 'tpr': 0.6122448979591837}

=== First 20 diffs ===
[{'bibcode': '2022ApJ...937L...3B',
  'mission': 'HST',
  'run': {'human_label': 'SCIENCE',
          'human_raw': 'SCIENCE',
          'llm_label': 'NONSCIENCE',
          'llm_raw': 'IGNORED'},
  'single': {'human_label': 'SCIENCE',
             'human_raw': 'SCIENCE',
             'llm_label'

In [ ]:
from collections import Counter

print("single_metrics", result["single_metrics"])
print("run_metrics", result["run_metrics"])
print("diff_count", len(result["diffs"]))
print("issue_counts", Counter(diff.get("issue", "label mismatch") for diff in result["diffs"]))
print("first_five_diffs")
for item in result["diffs"][:5]:
    print(item)

single_metrics {'tn': 67, 'fp': 6, 'fn': 17, 'tp': 30, 'tnr': 0.9178082191780822, 'fpr': 0.0821917808219178, 'fnr': 0.3617021276595745, 'tpr': 0.6382978723404256, 'precision': 0.8333333333333334, 'recall': 0.6382978723404256, 'f1': 0.7228915662650603, 'accuracy': 0.8083333333333333}
run_metrics {'tn': 66, 'fp': 5, 'fn': 19, 'tp': 30, 'tnr': 0.9295774647887324, 'fpr': 0.07042253521126761, 'fnr': 0.3877551020408163, 'tpr': 0.6122448979591837, 'precision': 0.8571428571428571, 'recall': 0.6122448979591837, 'f1': 0.7142857142857143, 'accuracy': 0.8}
diff_count 20
issue_counts Counter({'label mismatch': 20})
first_five_diffs
{'bibcode': '2022ApJ...937L...3B', 'mission': 'HST', 'single': {'human_raw': 'SCIENCE', 'llm_raw': 'MENTION', 'human_label': 'SCIENCE', 'llm_label': 'NONSCIENCE'}, 'run': {'human_raw': 'SCIENCE', 'llm_raw': 'IGNORED', 'human_label': 'SCIENCE', 'llm_label': 'NONSCIENCE'}}
{'bibcode': '2022ExA....53..547K', 'mission': 'HST', 'single': {'human_raw': 'MENTION', 'llm_raw': 'S

In [ ]:
from bibcat.llm.llm_io import get_source

bibcodes_to_check = [
    "2022JApA...43...86S",
    "2022ExA....53..547K",
]

for bibcode in bibcodes_to_check:
    print("\n===", bibcode, "===")
    print("summary human", eval_data[bibcode].get("human"))
    print("summary llm", eval_data[bibcode].get("llm"))
    source_item = get_source(bibcode=bibcode)
    print("source class_missions", source_item.get("class_missions") if source_item else None)
    runs = llm_runs_data.get(bibcode, [])
    print("run 0 raw output", runs[0] if runs else None)


=== 2022JApA...43...86S ===
summary human None
summary llm None
source class_missions {'HST': {'bibcode': '2022JApA...43...86S', 'papertype': 'MENTION'}, 'JWST': {'bibcode': '2022JApA...43...86S', 'papertype': 'MENTION'}}
run 0 raw output {'notes': 'The paper discusses the India-TMT project, which focuses on the Thirty Meter Telescope (TMT), a ground-based extremely large telescope. The TMT is not a MAST mission and does not appear in the provided list. The paper does mention JWST as a mission whose exoplanet targets may be followed up by TMT, but JWST data is only mentioned in a context of future complementary observations, not that this paper uses JWST data. The primary focus is on TMT instrumentation and developments, with no use of MAST mission data (HST or JWST). There is no indication of direct data usage or detailed scientific analysis involving MAST missions. Therefore, no MAST mission relevance or classification applies.', 'missions': []}

=== 2022ExA....53..547K ===
summary 

In [ ]:
binary_affecting = [
    diff
    for diff in result["diffs"]
    if diff.get("single", {}).get("human_label") != diff.get("run", {}).get("human_label")
    or diff.get("single", {}).get("llm_label") != diff.get("run", {}).get("llm_label")
]
print("binary_affecting_diff_count", len(binary_affecting))
for item in binary_affecting[:10]:
    print(item)

binary_affecting_diff_count 11
{'bibcode': '2022ExA....53..547K', 'mission': 'HST', 'single': {'human_raw': 'MENTION', 'llm_raw': 'SCIENCE', 'human_label': 'NONSCIENCE', 'llm_label': 'SCIENCE'}, 'run': {'human_raw': 'MENTION', 'llm_raw': 'MENTION', 'human_label': 'NONSCIENCE', 'llm_label': 'NONSCIENCE'}}
{'bibcode': '2022RAA....22k5007W', 'mission': 'HST', 'single': {'human_raw': 'SCIENCE', 'llm_raw': 'MENTION', 'human_label': 'SCIENCE', 'llm_label': 'NONSCIENCE'}, 'run': {'human_raw': 'SCIENCE', 'llm_raw': 'SCIENCE', 'human_label': 'SCIENCE', 'llm_label': 'SCIENCE'}}
{'bibcode': '2022RMxAA..58..133B', 'mission': 'HST', 'single': {'human_raw': 'SCIENCE', 'llm_raw': 'SCIENCE', 'human_label': 'SCIENCE', 'llm_label': 'SCIENCE'}, 'run': {'human_raw': 'SCIENCE', 'llm_raw': 'IGNORED', 'human_label': 'SCIENCE', 'llm_label': 'NONSCIENCE'}}
{'bibcode': '2023A&A...672A...3D', 'mission': 'HST', 'single': {'human_raw': 'SCIENCE', 'llm_raw': 'IGNORED', 'human_label': 'SCIENCE', 'llm_label': 'NONSCI